# RF 실험 2 · 누적 흐름 50지점 vs 100지점 (Closed-world)

**실험 2만 실행합니다. 실험 1의 그룹 단독 학습·Open-world는 실행하지 않습니다.**
`cumul100_mon_trim0.csv`를 `record_id`로 결합해 g6 50개만 100개로 교체합니다.
기존 g1~g5, monitored 기록, 공유 분할·CV fold, RF 설정을 고정합니다.

| 구성 | 50지점 | 100지점 |
| --- | --- | --- |
| 전체 (기본 비교) | g1~g5 60개 + g6 50개 = 110개 | g1~g5 60개 + g6 100개 = 160개 |
| g1+g6 (LR 보고서와 맞춘 추가 비교) | 53개 | 103개 |

- RF: 트리 100 / 깊이 20 / sqrt / 최소 리프 1 / class_weight=None / seed 42.
- 표준화·로그 변환·상수 제거·하이퍼파라미터 재탐색 없이, 같은 층화 5-fold로 비교합니다.
- LR의 해상도 표와 같이 **학습 데이터 내부 CV 결과만 비교**합니다. test는 학습·평가·특징 선택에 사용하지 않습니다.
- 기본 4조건 × 5-fold = 최대 20회 학습. `RUN_G1_G6=False`면 전체 2조건 × 5-fold = 최대 10회입니다.
- 기존 실험 1의 full 50지점 완료 결과가 실제 데이터·모델·fold·버전까지 일치할 때만 재사용합니다.
  일치하는 결과가 없으면 이 실험의 50지점 기준을 계산합니다. 이전 점수를 숫자로 하드코딩하지 않습니다.
- 완료 fold 저장·재사용, 조건별 시작·완료 로그 및 5분 간격 진행 표시를 제공합니다.

**준비 파일(모두 필요):** `features_mon_trim0.csv`, `cumul100_mon_trim0.csv`, 기존 `split_seed42.csv`.
unmon 데이터는 필요 없습니다. 100지점 파일은 기존 50지점을 단순 보간한 값으로 대체하지 않습니다.

0번 경로를 확인하고 위에서부터 실행하세요. 이 파일에는 실제 100지점 데이터의 학습 결과가 미리 들어 있지 않습니다.


## 0. 경로와 고정 설정

100지점 CSV가 다른 폴더에 있으면 `CUMUL100_PATH`만 수정하세요.
기본적으로 `g6_…`, `cumul_…`, `cumul100_…`, 또는 숫자 이름의 100개 열을 식별합니다.
자동 식별이 안 되는 파일은 오류에 표시되는 열 이름을 확인하고 `CUMUL100_COLUMNS`에 실제 100개 열을 순서대로 지정하세요.


In [1]:
FEATURE_DIR = "/content/drive/MyDrive/MLB-Project/feature"
MON50_PATH = FEATURE_DIR + "/features_mon_trim0.csv"
CUMUL100_PATH = FEATURE_DIR + "/cumul100_mon_trim0.csv"
RESULT_ROOT = "/content/drive/MyDrive/MLB-Project/results"
SPLIT_PATH = RESULT_ROOT + "/split_seed42.csv"
RESULT_DIR = RESULT_ROOT + "/rf_experiment2_cumul_resolution"
GROUP_ONLY_RESULT_DIR = RESULT_ROOT + "/rf_experiment1_group_only"

RUN_G1_G6 = True  # LR과 같은 4조건. False면 전체 110/160개만 비교합니다.
REUSE_GROUP_ONLY_BASELINE = True
USE_CACHE = True
CUMUL100_COLUMNS = None  # 자동 식별 불가 시 실제 100개 열 이름 리스트로 지정
SEED = 42
N_FOLDS = 5
N_JOBS = -1
EXPECTED_MON_CLASSES = 95
LOG_INTERVAL_SECONDS = 300
RF_PARAMS = dict(n_estimators=100, max_depth=20, max_features="sqrt",
                 min_samples_leaf=1, class_weight=None)


In [2]:
import os,sys,re,json,time,tempfile,threading,platform
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from joblib import hash as joblib_hash
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")

if not isinstance(N_FOLDS,int) or N_FOLDS < 2 or LOG_INTERVAL_SECONDS <= 0:
    raise ValueError("N_FOLDS와 LOG_INTERVAL_SECONDS 설정을 확인하세요.")
if RF_PARAMS["max_features"] != "sqrt":
    raise ValueError("기존 RF와 비교하려면 max_features='sqrt'를 유지하세요.")
VERSIONS = dict(python=platform.python_version(), sklearn=sklearn.__version__,
                numpy=np.__version__, pandas=pd.__version__)
def make_model():
    return RandomForestClassifier(**RF_PARAMS,random_state=SEED,n_jobs=N_JOBS)
pd.set_option("display.max_columns",20)
print("Closed-world만 실행 | scikit-learn",sklearn.__version__,"| 고정 RF:",RF_PARAMS)


Mounted at /content/drive
Closed-world만 실행 | scikit-learn 1.6.1 | 고정 RF: {'n_estimators': 100, 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'class_weight': None}


## 1. 두 CSV 점검 · ID로 100지점 결합

100지점 파일의 행 순서가 달라도 `record_id`로 원래 mon CSV 순서에 맞춥니다.
원래 mon에 필요한 ID가 하나라도 없거나, ID가 중복되거나, 라벨·trim 등의 메타데이터가 충돌하면 학습 전에 중단합니다.
100지점 파일에 g1~g5가 함께 있더라도 모델에는 **원래 50지점 CSV의 g1~g5 값만** 사용합니다.


In [3]:
INFO_COLS = ["record_id","dataset","page_id","sample_idx","label","trim"]
GROUP_SIZES = {"g1":3,"g2":17,"g3":6,"g4":3,"g5":31,"g6":50}

def choose_100_columns(columns):
    if CUMUL100_COLUMNS is not None:
        cols = list(CUMUL100_COLUMNS)
    else:
        candidates = [
            [c for c in columns if c.startswith("g6_")],
            [c for c in columns if re.fullmatch(r"(?:cumul|cumul100)_?\d+",c)],
            [c for c in columns if re.fullmatch(r"\d+",c)],
        ]
        cols = next((c for c in candidates if len(c)==100),None)
        if cols is None:
            raise ValueError("100개 누적 특징 열을 자동 판별할 수 없습니다. CUMUL100_COLUMNS에 실제 열 100개를 지정하세요.\n"
                             + str(list(columns)))
        matches = [re.search(r"(\d+)$",c) for c in cols]
        if not all(matches):
            raise ValueError("지점 순서를 자동 판별할 수 없습니다. CUMUL100_COLUMNS에 순서대로 100개 열을 지정하세요.")
        ids = [int(m.group(1)) for m in matches]
        if set(ids) not in (set(range(100)),set(range(1,101))):
            raise ValueError("100지점 번호가 0~99 또는 1~100이 아닙니다. 실제 지점 순서를 확인하고 CUMUL100_COLUMNS를 지정하세요.")
        cols = [c for _,c in sorted(zip(ids,cols))]
    if len(cols)!=100 or len(set(cols))!=100 or not set(cols).issubset(columns):
        raise ValueError("CUMUL100_COLUMNS에는 CSV에 존재하는 서로 다른 특징 열 100개가 필요합니다.")
    if set(cols)&set(INFO_COLS+["split","y"]):
        raise ValueError("ID·라벨 등 정보 열을 특징으로 지정할 수 없습니다.")
    return cols

def load_resolution_data(mon50_path,cumul100_path):
    for path in [mon50_path,cumul100_path]:
        if not Path(path).is_file():
            raise FileNotFoundError(f"필요한 CSV가 없습니다: {path}\n0번 경로 설정을 확인하세요.")
    t = time.perf_counter()
    base = pd.read_csv(mon50_path,dtype={"record_id":str})
    if not set(INFO_COLS).issubset(base):
        raise ValueError("50지점 CSV의 정보 열 6개를 확인하세요.")
    features = [c for c in base if any(c.startswith(g+"_") for g in GROUP_SIZES)]
    actual = {g:sum(c.startswith(g+"_") for c in features) for g in GROUP_SIZES}
    if actual!=GROUP_SIZES or set(base.columns)!=set(INFO_COLS+features):
        raise ValueError(f"기존 110특징 CSV가 필요합니다. 실제 구성: {actual}")
    if base.record_id.isna().any() or base.record_id.duplicated().any():
        raise ValueError("50지점 CSV의 ID 누락/중복")
    if not base.dataset.eq("mon").all() or not base.trim.eq(0).all():
        raise ValueError("monitored trim0 데이터만 사용합니다.")
    if base.label.isna().any() or not np.equal(base.label,np.floor(base.label)).all():
        raise ValueError("mon 라벨은 정수 사이트 번호여야 합니다.")
    base["label"] = base.label.astype(int)
    if sorted(base.label.unique())!=list(range(EXPECTED_MON_CLASSES)):
        raise ValueError(f"mon 라벨 0~{EXPECTED_MON_CLASSES-1}을 기대합니다.")
    if not np.isfinite(base[features].to_numpy(dtype=float)).all():
        raise ValueError("50지점 특징에 NaN/inf가 있습니다.")

    raw = pd.read_csv(cumul100_path,dtype={"record_id":str})
    if "record_id" not in raw or raw.record_id.isna().any() or raw.record_id.duplicated().any():
        raise ValueError("100지점 CSV에 고유한 record_id가 필요합니다.")
    cols100 = choose_100_columns(raw.columns)
    missing = set(base.record_id)-set(raw.record_id)
    if missing:
        raise ValueError(f"100지점 CSV에 기존 mon ID {len(missing)}개가 없습니다. 일부 예: {sorted(missing)[:5]}")
    aligned = raw.set_index("record_id").loc[base.record_id].reset_index()
    for col in ["dataset","page_id","sample_idx","label","trim"]:
        if col in aligned and not np.array_equal(aligned[col].to_numpy(),base[col].to_numpy()):
            raise ValueError(f"같은 record_id의 {col} 값이 서로 다릅니다.")
    values100 = aligned[cols100].to_numpy(dtype=float)
    if not np.isfinite(values100).all():
        raise ValueError("100지점 특징에 NaN/inf가 있습니다.")
    other = [c for c in features if not c.startswith("g6_")]
    # 원래 g6 50개는 제거하고, 원래 g1~g5 60개에 실제 100지점 값만 결합합니다.
    new_names = [f"g6_cumul100_{i:02d}" for i in range(100)]
    expanded = pd.concat([base[INFO_COLS+other].reset_index(drop=True),
                          pd.DataFrame(values100,columns=new_names)],axis=1)
    assert np.array_equal(expanded[other].to_numpy(),base[other].to_numpy())
    assert expanded.record_id.equals(base.record_id)
    audit = dict(n_mon=len(base),n_raw_100=len(raw),n_ignored_extra_ids=len(set(raw.record_id)-set(base.record_id)),
                 join_time_s=time.perf_counter()-t,non_g6_features=len(other),
                 original_features=len(features),expanded_features=len(other+new_names))
    return base,expanded,features,other+new_names,cols100,new_names,audit

mon50,mon100,FEATS50,FEATS100,SOURCE100,NEW100,DATA_AUDIT = load_resolution_data(MON50_PATH,CUMUL100_PATH)
assert len(FEATS50)==110 and len(FEATS100)==160
print(f"mon {len(mon50):,}건: ID 정렬·메타데이터·고정 특징 60개 확인 완료")
print("입력 100지점 특징:",SOURCE100[:3],"…",SOURCE100[-3:])
display(pd.DataFrame([DATA_AUDIT]))


mon 19,000건: ID 정렬·메타데이터·고정 특징 60개 확인 완료
입력 100지점 특징: ['g6_cumul100_000', 'g6_cumul100_001', 'g6_cumul100_002'] … ['g6_cumul100_097', 'g6_cumul100_098', 'g6_cumul100_099']


,n_mon,n_raw_100,n_ignored_extra_ids,join_time_s,non_g6_features,original_features,expanded_features
0,19000,19000,0,3.050749,60,110,160


## 2. 기존 공유 분할 · 동일한 학습 fold

기존 `split_seed42.csv`를 반드시 사용합니다. 새 분할은 만들지 않습니다.
mon 원래 행 순서를 유지하고 50지점/100지점 모두 동일한 fold 인덱스로 학습합니다.
test ID도 보관하지만 이 실험에서는 test를 평가하지 않습니다.


In [4]:
if not Path(SPLIT_PATH).is_file():
    raise FileNotFoundError(f"기존 공유 분할 파일이 필요합니다: {SPLIT_PATH}")
split = pd.read_csv(SPLIT_PATH,dtype={"record_id":str})
if not {"record_id","split"}.issubset(split):
    raise ValueError("분할 파일에 record_id, split 열이 필요합니다.")
if (split.record_id.isna().any() or split.record_id.duplicated().any()
        or not split.split.isin(["train","test"]).all()):
    raise ValueError("분할 파일의 ID 중복/누락과 train/test 값을 확인하세요.")
if not set(mon50.record_id).issubset(set(split.record_id)):
    raise ValueError("공유 분할에 없는 mon 기록이 있습니다.")
lookup = split.set_index("record_id")["split"]
mon50 = mon50.assign(split=mon50.record_id.map(lookup),y=mon50.label)
mon100 = mon100.assign(split=mon100.record_id.map(lookup),y=mon100.label)
if set(mon50.split)!={"train","test"}:
    raise ValueError("공유 분할에 mon 학습·test 기록이 모두 있어야 합니다.")
TRAIN = {50:mon50.loc[mon50.split.eq("train")].reset_index(drop=True),
         100:mon100.loc[mon100.split.eq("train")].reset_index(drop=True)}
TEST50 = mon50.loc[mon50.split.eq("test")].reset_index(drop=True)  # 기존 캐시의 데이터 식별에만 사용
tr50 = TRAIN[50]
if tr50.y.value_counts().min()<N_FOLDS or set(tr50.y)!=set(TEST50.y):
    raise ValueError("클래스 누락 또는 CV 학습 표본 부족")
if not TRAIN[50][["record_id","y"]].equals(TRAIN[100][["record_id","y"]]):
    raise ValueError("50/100지점 학습 기록 또는 정답 순서가 다릅니다.")
FOLDS = list(StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED).split(tr50[FEATS50],tr50.y))
fold_ids = np.full(len(tr50),-1)
for k,(_,iv) in enumerate(FOLDS,1):fold_ids[iv]=k
FEATURE_SETS = {
    ("full",50):FEATS50, ("full",100):FEATS100,
    ("g1_g6",50):[c for c in FEATS50 if c.startswith(("g1_","g6_"))],
    ("g1_g6",100):[c for c in FEATS100 if c.startswith(("g1_","g6_"))],
}
COMPOSITIONS = ["full"]+(["g1_g6"] if RUN_G1_G6 else [])
print("mon 학습:",len(tr50),"| test(평가하지 않음):",len(TEST50))
print(f"{len(COMPOSITIONS)*2}조건 × {N_FOLDS}-fold = 최대 {len(COMPOSITIONS)*2*N_FOLDS}회 학습 (재사용 시 감소)")


mon 학습: 15200 | test(평가하지 않음): 3800
4조건 × 5-fold = 최대 20회 학습 (재사용 시 감소)


## 3. 재개 가능한 CV · 기존 50지점 기준 재사용

5개 fold를 순차 실행하고 RF 내부에서 CPU를 병렬 사용합니다. 완료 fold는 즉시 저장합니다.
기존 Group-only의 full 50지점 결과는 동일한 실제 입력·정답·fold·모델·라이브러리 버전의 식별값과
OOF 예측·fold 점수가 일치할 때만 재사용합니다. 일치 여부를 확인할 수 없으면 새로 학습합니다.
재사용한 시간은 원래 계산 당시 시간이며 이번 세션 경과 시간과 다릅니다.


In [5]:
def atomic_json(path,payload):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.NamedTemporaryFile("w",encoding="utf-8",dir=path.parent,delete=False,suffix=".tmp") as f:
        temp=Path(f.name);json.dump(payload,f,ensure_ascii=False,allow_nan=False,indent=2)
    os.replace(temp,path)
def atomic_csv(frame,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.NamedTemporaryFile("w",encoding="utf-8-sig",dir=path.parent,delete=False,suffix=".tmp",newline="") as f:
        temp=Path(f.name);frame.to_csv(f,index=False)
    os.replace(temp,path)
def atomic_npz(path,**arrays):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.NamedTemporaryFile("wb",dir=path.parent,delete=False,suffix=".tmp") as f:
        temp=Path(f.name);np.savez_compressed(f,**arrays)
    os.replace(temp,path)

RUN_ID = joblib_hash(("rf_resolution_cv_v1",mon50[INFO_COLS+FEATS50+["split"]],
                      mon100[INFO_COLS+FEATS100+["split"]],FOLDS,make_model().get_params(),VERSIONS))
RUN_DIR = Path(RESULT_DIR)/"runs"/RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=True)
atomic_json(RUN_DIR/"config.json",dict(run_id=RUN_ID,model=make_model().get_params(),versions=VERSIONS,
    seed=SEED,n_folds=N_FOLDS,mon50_path=MON50_PATH,cumul100_path=CUMUL100_PATH,
    split_path=SPLIT_PATH,group_only_results=GROUP_ONLY_RESULT_DIR,test_evaluated=False,
    data_audit=DATA_AUDIT))
atomic_csv(mon50[["record_id","label","split"]],RUN_DIR/"split_used.csv")
atomic_csv(tr50[["record_id","y"]].assign(cv_fold=fold_ids),RUN_DIR/"cv_assignment.csv")
atomic_csv(pd.DataFrame([dict(cumul_points=100,source_column=s,model_feature=n)
                         for s,n in zip(SOURCE100,NEW100)]),RUN_DIR/"cumul100_column_map.csv")

def find_previous_full50():
    if not REUSE_GROUP_ONLY_BASELINE:return None
    # 실제 실행된 clean Group-only 노트북과 같은 식별 규칙입니다.
    sig = joblib_hash(("rf_group_only_v1","closed_world","full",FEATS50,
        tr50[["record_id","y"]+FEATS50],TEST50[["record_id","y"]+FEATS50],FOLDS,
        make_model().get_params(),VERSIONS))
    root=Path(GROUP_ONLY_RESULT_DIR)/"runs"
    for marker in sorted(root.glob(f"*/closed_world__full__{sig[:12]}/complete.json")):
        try:
            saved=json.loads(marker.read_text(encoding="utf-8"))
            if saved.get("signature")!=sig:continue
            folder=marker.parent
            folds=pd.read_csv(folder/"cv_folds.csv")
            oof=pd.read_csv(folder/"oof_predictions.csv",dtype={"record_id":str})
            if oof.record_id.duplicated().any() or set(oof.record_id)!=set(tr50.record_id):continue
            oof=oof.set_index("record_id").loc[tr50.record_id].reset_index()
            if not np.array_equal(oof.y,tr50.y) or not np.array_equal(oof.cv_fold,fold_ids):continue
            if set(folds.fold)!=set(range(1,N_FOLDS+1)) or len(folds)!=N_FOLDS:continue
            for k,(_,iv) in enumerate(FOLDS,1):
                r=folds.set_index("fold").loc[k]
                if not np.isclose(accuracy_score(tr50.y.iloc[iv],oof.pred.iloc[iv]),r.acc,rtol=0,atol=1e-12):
                    raise ValueError("기존 OOF 예측과 fold 점수 불일치")
            if not np.isclose(folds.acc.mean(),saved["row"]["cv_acc"],rtol=0,atol=1e-12):continue
            return dict(folder=str(folder),folds=folds.set_index("fold"),oof=oof)
        except (OSError,ValueError,KeyError,TypeError):
            continue
    return None

PREVIOUS_FULL50 = find_previous_full50()
print("기존 full/50 기준:","정합성 확인 후 재사용 가능" if PREVIOUS_FULL50 else "호환 완료 결과 없음 → 이 실험에서 계산")
RESULTS={};FOLD_TABLES={}

def summary_frame():
    frame=pd.DataFrame(RESULTS.values())
    if frame.empty:return frame
    frame["delta_vs_50"]=np.nan
    for composition in frame.composition.unique():
        base=frame.loc[(frame.composition==composition)&(frame.cumul_points==50)]
        if not base.empty:
            mask=frame.composition==composition
            frame.loc[mask,"delta_vs_50"]=frame.loc[mask,"cv_acc"]-base.iloc[0].cv_acc
    return frame

def run_resolution(composition,points):
    data=TRAIN[points];cols=FEATURE_SETS[(composition,points)]
    X=data[cols];y=data.y.to_numpy()
    signature=joblib_hash(("rf_resolution_condition_v1",composition,points,
                           data[["record_id","y"]+cols],FOLDS,make_model().get_params(),VERSIONS))
    folder=RUN_DIR/f"{composition}_cumul{points}_{signature[:12]}"
    folder.mkdir(parents=True,exist_ok=True)
    atomic_json(folder/"condition.json",dict(signature=signature,composition=composition,points=points,
                                           columns=cols,model=make_model().get_params()))
    state={"done":0};stop=threading.Event();started=time.perf_counter()
    def heartbeat():
        while not stop.wait(LOG_INTERVAL_SECONDS):
            print(f"[진행] {composition}/{points}: {state['done']}/{N_FOLDS} fold 완료 | {(time.perf_counter()-started)/60:.1f}분",flush=True)
    reporter=threading.Thread(target=heartbeat,daemon=True);reporter.start()
    records=[];oof=np.empty(len(data),dtype=int)
    print(f"[시작] {composition}/g6 {points}지점 ({len(cols)}특징): {N_FOLDS}-fold",flush=True)
    try:
        for k,(it,iv) in enumerate(FOLDS,1):
            cache=folder/f"fold_{k}.npz"
            if USE_CACHE and cache.is_file():
                with np.load(cache,allow_pickle=False) as s:
                    if str(s["signature"].item())!=signature or not np.array_equal(s["validation_index"],iv):
                        raise ValueError("fold 캐시 식별값 또는 인덱스 불일치")
                    pred=s["pred"].copy();fit_s=float(s["fit_s"]);predict_s=float(s["predict_s"])
                source="experiment2_cache"
            elif composition=="full" and points==50 and PREVIOUS_FULL50 is not None:
                previous=PREVIOUS_FULL50
                pred=previous["oof"].pred.to_numpy()[iv]
                record=previous["folds"].loc[k]
                fit_s=float(record.fit_s);predict_s=float(record.predict_s)
                source="group_only_full50"
            else:
                model=make_model();t=time.perf_counter()
                model.fit(X.iloc[it],y[it]);fit_s=time.perf_counter()-t
                t=time.perf_counter();pred=model.predict(X.iloc[iv]);predict_s=time.perf_counter()-t
                del model;source="computed_now"
            if len(pred)!=len(iv) or not set(np.unique(pred)).issubset(set(y)):
                raise ValueError("검증 예측의 길이 또는 라벨 불일치")
            if USE_CACHE and source!="experiment2_cache":
                atomic_npz(cache,signature=np.array(signature),validation_index=iv,pred=pred,
                           fit_s=np.array(fit_s),predict_s=np.array(predict_s))
            oof[iv]=pred
            records.append(dict(fold=k,acc=float(accuracy_score(y[iv],pred)),
                macro_f1=float(f1_score(y[iv],pred,average="macro",zero_division=0)),
                n_train=len(it),n_validation=len(iv),fit_s=fit_s,predict_s=predict_s,source=source))
            state["done"]=k
        frame=pd.DataFrame(records)
        atomic_csv(frame,folder/"cv_folds.csv")
        atomic_csv(data[["record_id","label","y"]].assign(cv_fold=fold_ids,pred=oof),folder/"oof_predictions.csv")
        FOLD_TABLES[(composition,points)]=frame
        row=dict(composition=composition,cumul_points=points,n_features=len(cols),
                 cv_acc=float(frame.acc.mean()),cv_std=float(frame.acc.std(ddof=0)),
                 cv_macro_f1=float(frame.macro_f1.mean()),cv_fit_time_s=float(frame.fit_s.sum()),
                 cv_predict_time_s=float(frame.predict_s.sum()),session_elapsed_s=time.perf_counter()-started,
                 cached_folds=int(frame.source.ne("computed_now").sum()),
                 result_folder=str(folder),signature=signature)
        RESULTS[(composition,points)]=row
        atomic_csv(summary_frame(),RUN_DIR/"rf_g6_resolution.csv")
        print(f"[완료] {composition}/{points}: CV={row['cv_acc']:.4f} ± {row['cv_std']:.4f} | 재사용 {row['cached_folds']}/{N_FOLDS} fold | 이번 경과 {row['session_elapsed_s']/60:.1f}분",flush=True)
        return row
    finally:
        stop.set();reporter.join(timeout=1)

print("실험 2 결과 폴더:",RUN_DIR)


기존 full/50 기준: 정합성 확인 후 재사용 가능
실험 2 결과 폴더: /content/drive/MyDrive/MLB-Project/results/rf_experiment2_cumul_resolution/runs/e4f9e355d8bbbf27751f246c2d437d45


## 4. 실험 2 실행

최대 4조건의 CV만 계산합니다. 최적 설정 탐색, 실험 1의 그룹 단독 학습, Open-world 학습은 하지 않습니다.


In [6]:
for composition in COMPOSITIONS:
    for points in [50,100]:
        run_resolution(composition,points)
display(summary_frame().round(4))


[시작] full/g6 50지점 (110특징): 5-fold
[완료] full/50: CV=0.8476 ± 0.0068 | 재사용 5/5 fold | 이번 경과 0.0분
[시작] full/g6 100지점 (160특징): 5-fold
[완료] full/100: CV=0.8476 ± 0.0068 | 재사용 0/5 fold | 이번 경과 3.3분
[시작] g1_g6/g6 50지점 (53특징): 5-fold
[완료] g1_g6/50: CV=0.7918 ± 0.0084 | 재사용 0/5 fold | 이번 경과 2.3분
[시작] g1_g6/g6 100지점 (103특징): 5-fold
[완료] g1_g6/100: CV=0.7998 ± 0.0042 | 재사용 0/5 fold | 이번 경과 3.3분


,composition,cumul_points,n_features,cv_acc,cv_std,cv_macro_f1,cv_fit_time_s,cv_predict_time_s,session_elapsed_s,cached_folds,result_folder,signature,delta_vs_50
0,full,50,110,0.8476,0.0068,0.8464,118.1834,0.6176,0.2086,5,/content/drive/MyDrive/MLB-Project/results/rf_...,b387d1c3c241b83ce1b115c9c15af822,0.000
1,full,100,160,0.8476,0.0068,0.8463,197.1566,0.6846,198.1274,0,/content/drive/MyDrive/MLB-Project/results/rf_...,e8ea668bcf4cf70a9b389e86e9747dc6,0.000
2,g1_g6,50,53,0.7918,0.0084,0.7892,134.1795,0.5497,135.0045,0,/content/drive/MyDrive/MLB-Project/results/rf_...,fc950ee2905fe0dfed25fc286c4ad1ea,0.000
3,g1_g6,100,103,0.7998,0.0042,0.7975,194.7453,0.7265,195.7857,0,/content/drive/MyDrive/MLB-Project/results/rf_...,fa35693472b18a7423f9a99ce4bd3344,0.008


## 5. 비교 표 · 같은 fold 차이 · 보고서 저장

`delta_vs_50`은 100지점 CV 정확도에서 같은 구성의 50지점 정확도를 뺀 값입니다.
각 행의 표준편차는 fold 간 변동이며 신뢰구간이 아닙니다. 같은 fold에서의 차이도 함께 보세요.
특징 수를 늘린 효과가 모든 모델·과제에서 재현된다고 확대하지 않습니다.


In [7]:
summary=summary_frame();paired=[];rows=[]
for composition in COMPOSITIONS:
    a=RESULTS[(composition,50)];b=RESULTS[(composition,100)]
    diff=FOLD_TABLES[(composition,100)].merge(FOLD_TABLES[(composition,50)],on="fold",suffixes=("_100","_50"),validate="one_to_one")
    diff["delta_acc"]=diff.acc_100-diff.acc_50
    for _,r in diff.iterrows():
        paired.append(dict(composition=composition,fold=int(r.fold),acc_50=r.acc_50,acc_100=r.acc_100,delta_acc=r.delta_acc))
    rows.append(dict(composition=composition,n_features_50=a["n_features"],n_features_100=b["n_features"],
        cv_50=f"{a['cv_acc']:.4f} ± {a['cv_std']:.4f}",cv_100=f"{b['cv_acc']:.4f} ± {b['cv_std']:.4f}",
        delta_acc=b["cv_acc"]-a["cv_acc"],positive_folds=int(diff.delta_acc.gt(0).sum()),
        cv_fit_s_50=a["cv_fit_time_s"],cv_fit_s_100=b["cv_fit_time_s"],
        cv_predict_s_50=a["cv_predict_time_s"],cv_predict_s_100=b["cv_predict_time_s"]))
comparison=pd.DataFrame(rows)
atomic_csv(summary,RUN_DIR/"rf_g6_resolution.csv")
atomic_csv(comparison,RUN_DIR/"rf_g6_resolution_comparison.csv")
atomic_csv(pd.DataFrame(paired),RUN_DIR/"paired_fold_differences.csv")
display(comparison.round(4))

lines=["# RF 실험 2 · 누적 흐름 해상도 비교", "", "Closed-world / trim0 / 기존 공유 분할 / 고정 RF / 학습 데이터 내부 CV만 평가.",
       "",f"RF: {json.dumps(RF_PARAMS,ensure_ascii=False)}, seed={SEED}, {N_FOLDS}-fold.","",
       "| 구성 | 특징 수 50→100 | CV 50지점 | CV 100지점 | 변화 |", "|---|---|---|---|---|"]
for r in rows:
    lines.append(f"| {r['composition']} | {r['n_features_50']}→{r['n_features_100']} | {r['cv_50']} | {r['cv_100']} | {r['delta_acc']:+.4f} |")
lines += ["", "## 해석", ""]
for r in rows:
    lines.append(f"- {r['composition']}: 100지점−50지점 평균 차이 {r['delta_acc']:+.4f}, 개선된 fold {r['positive_folds']}/{N_FOLDS}개. 동일 fold의 차이는 paired_fold_differences.csv 참고.")
lines += ["- 표준편차는 신뢰구간이 아닙니다. 일부 CV 차이만으로 통계적 유의성이나 전체 특징의 충분성을 주장하지 않습니다.",
          "- RF 설정을 고정한 비교이므로 각 해상도에서 최적 설정을 다시 찾은 결과는 아닙니다.",
          "- sqrt 규칙은 같지만 분기 특징 수는 전체 10→12, g1+g6 7→10으로 달라집니다.",
          "- monitored 100지점만 사용했으므로 Open-world 해상도 결과로 해석하지 않습니다.",
          "- 기록한 fit/predict 시간은 fold 합계입니다. 캐시 재사용 행은 원래 계산 당시 시간이며 이번 세션 경과 시간과 다릅니다.",
          "- 기존 50지점 기준을 유지하고, 개선이 관찰된 구성은 후속 모델 비교 후보로 검토합니다."]
(RUN_DIR/"rf_g6_resolution_report.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
print("저장 완료:",RUN_DIR)


,composition,n_features_50,n_features_100,cv_50,cv_100,delta_acc,positive_folds,cv_fit_s_50,cv_fit_s_100,cv_predict_s_50,cv_predict_s_100
0,full,110,160,0.8476 ± 0.0068,0.8476 ± 0.0068,0.000,2,118.1834,197.1566,0.6176,0.6846
1,g1_g6,53,103,0.7918 ± 0.0084,0.7998 ± 0.0042,0.008,4,134.1795,194.7453,0.5497,0.7265


저장 완료: /content/drive/MyDrive/MLB-Project/results/rf_experiment2_cumul_resolution/runs/e4f9e355d8bbbf27751f246c2d437d45


## 저장 파일

`RESULT_DIR/runs/<실행 ID>/` 아래에 저장합니다.

| 파일 | 내용 |
| --- | --- |
| `rf_g6_resolution.csv` | 조건별 CV·표준편차·시간·캐시 재사용 기록 |
| `rf_g6_resolution_comparison.csv` | 50/100 비교 표와 변화량 |
| `rf_g6_resolution_report.md` | 보고서에 옮길 요약 |
| `paired_fold_differences.csv` | 동일 fold의 100−50 차이 |
| `config.json`, `split_used.csv`, `cv_assignment.csv` | 설정·분할·CV fold |
| `cumul100_column_map.csv` | 100지점 입력 열과 모델 입력 열의 대응 |
| 조건별 `cv_folds.csv`, `oof_predictions.csv` | fold 점수와 OOF 예측 |

**100지점 원본 추출 조건:** 파일명이 같다는 이유만으로 동일한 추출 정의가 보장되지는 않습니다.
trim·방향 부호·누적 흐름 정의 등은 기존 50지점 데이터와 같고 표본화 지점 수만 다르게 추출된 파일을 사용하세요.
코드가 ID·메타데이터·특징 개수를 검사하지만 추출 알고리즘 자체의 동일성까지 검증하지는 않습니다.

실제 100지점 CSV가 이 대화에 제공되지 않아, 제작 과정에서는 시험용 100열 데이터로 결합·분할·CV·캐시 동작을 검증합니다.
시험용 수치는 이 노트북에 실제 실험 결과로 포함하지 않습니다.
